# โปรเจกต์ Image Classification ด้วย CIFAR-10
## เปรียบเทียบประสิทธิภาพของ SVM, Random Forest และ CNN

งานนี้มีวัตถุประสงค์เพื่อเปรียบเทียบประสิทธิภาพของโมเดล 3 ประเภทในการจำแนกภาพจากชุดข้อมูล CIFAR-10 ได้แก่

1. SVM  
2. Random Forest  
3. CNN  

โดยจะทดลองหลายแนวทาง เช่น Raw Pixels, HOG, PCA และ Data Augmentation  
แล้วเปรียบเทียบผลด้วย Accuracy, Precision, Recall, F1-score และ Confusion Matrix

In [ ]:
# =========================
# 1) Import Libraries
# =========================
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from skimage.feature import hog
from skimage.color import rgb2gray

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# =========================
# 2) ตั้งค่า seed เพื่อให้ผลลัพธ์ reproducible
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# โหลดข้อมูล CIFAR-10

ในส่วนนี้จะโหลดชุดข้อมูล CIFAR-10 ทั้ง train และ test  
และแปลงออกมาเป็น numpy array เพื่อใช้กับโมเดล SVM และ Random Forest

In [ ]:
# =========================
# 3) Dataset
# ใช้ CIFAR-10 ตามโจทย์
# =========================
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Train shape:", x_train_full.shape)
print("Train label shape:", y_train_full.shape)
print("Test shape:", x_test.shape)
print("Test label shape:", y_test.shape)

In [ ]:
# =========================
# 4) แบ่ง train / validation
# เพื่อใช้ประเมินระหว่างการฝึก
# =========================
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_full
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

# EDA: สำรวจข้อมูลเบื้องต้น


In [ ]:
# =========================
# 5) EDA: ดูตัวอย่างภาพ
# สำรวจลักษณะเบื้องต้นของข้อมูล
# =========================
plt.figure(figsize=(14, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 6) EDA: ดูจำนวนข้อมูลแต่ละคลาส
# =========================
train_counts = pd.Series(y_train.flatten()).value_counts().sort_index()
eda_df = pd.DataFrame({
    "class_id": range(10),
    "class_name": class_names,
    "count": train_counts.values
})
display(eda_df)

plt.figure(figsize=(10, 4))
plt.bar(eda_df["class_name"], eda_df["count"])
plt.title("Class Distribution in Training Set")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# =========================
# 8) ฟังก์ชันช่วยสร้าง feature
# =========================

def flatten_images(x):
    # แปลงภาพ 32x32x3 ให้เป็นเวกเตอร์ 3072 มิติ
    return x.reshape(len(x), -1)

def extract_hog_features(images):
    # ใช้ HOG เพื่อดึงลักษณะด้านรูปร่าง/ขอบวัตถุ
    features = []
    for img in images:
        gray = rgb2gray(img)
        feat = hog(
            gray,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )
        features.append(feat)
    return np.array(features)

def extract_color_histogram(images, bins=8):
    # ใช้ Color Histogram เพื่อดึงลักษณะด้านสี
    features = []
    for img in images:
        hist_r, _ = np.histogram(img[:, :, 0], bins=bins, range=(0, 255), density=True)
        hist_g, _ = np.histogram(img[:, :, 1], bins=bins, range=(0, 255), density=True)
        hist_b, _ = np.histogram(img[:, :, 2], bins=bins, range=(0, 255), density=True)
        features.append(np.concatenate([hist_r, hist_g, hist_b]))
    return np.array(features)

In [ ]:
# =========================
# 9) ลดขนาดข้อมูลสำหรับ classical ML
# SVM / RF บน CIFAR-10 เต็มชุดใช้เวลานานมาก
# จึงสุ่ม subset เพื่อให้รันได้จริงในเครื่องทั่วไป
# =========================
CLASSICAL_TRAIN_SIZE = 12000
CLASSICAL_VAL_SIZE   = 3000
CLASSICAL_TEST_SIZE  = 3000

train_idx = np.random.choice(len(x_train), CLASSICAL_TRAIN_SIZE, replace=False)
val_idx   = np.random.choice(len(x_val), CLASSICAL_VAL_SIZE, replace=False)
test_idx  = np.random.choice(len(x_test), CLASSICAL_TEST_SIZE, replace=False)

x_train_cls = x_train[train_idx]
y_train_cls = y_train[train_idx].ravel()

x_val_cls = x_val[val_idx]
y_val_cls = y_val[val_idx].ravel()

x_test_cls = x_test[test_idx]
y_test_cls = y_test[test_idx].ravel()

print(x_train_cls.shape, y_train_cls.shape)
print(x_val_cls.shape, y_val_cls.shape)
print(x_test_cls.shape, y_test_cls.shape)

In [ ]:
# =========================
# 10) เตรียม feature ตามปัญหาที่ต้องการแก้
# =========================

# Problem 4: baseline -> raw pixels
X_train_raw = flatten_images(x_train_cls)
X_val_raw   = flatten_images(x_val_cls)
X_test_raw  = flatten_images(x_test_cls)

# Problem 1: high dimension -> PCA
pca = PCA(n_components=300, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_raw)
X_val_pca   = pca.transform(X_val_raw)
X_test_pca  = pca.transform(X_test_raw)

print("PCA explained variance ratio sum:", pca.explained_variance_ratio_.sum())

# Problem 2: shape matters -> HOG
X_train_hog = extract_hog_features(x_train_cls)
X_val_hog   = extract_hog_features(x_val_cls)
X_test_hog  = extract_hog_features(x_test_cls)

print("HOG shape:", X_train_hog.shape)

# Problem 3: color helps -> Color Histogram
X_train_color = extract_color_histogram(x_train_cls, bins=8)
X_val_color   = extract_color_histogram(x_val_cls, bins=8)
X_test_color  = extract_color_histogram(x_test_cls, bins=8)

print("ColorHist shape:", X_train_color.shape)

In [ ]:
# =========================
# 11) ฟังก์ชันประเมินผล
# =========================
from sklearn.metrics import precision_score, recall_score, f1_score

results = []

def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    results.append({
        "Experiment": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

    print(f"\n========== {model_name} ==========")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
    plt.title(f"Confusion Matrix: {model_name}")
    plt.show()

    print(f"\n========== {model_name} ==========")

In [ ]:
# =========================
# 12) Experiment 1: SVM_RawPixels
# Problem 4: baseline ด้วย pixel ดิบ
# =========================
svm_raw = SVC(kernel="rbf", C=5, gamma="scale", random_state=SEED)
svm_raw.fit(X_train_raw, y_train_cls)

pred_svm_raw = svm_raw.predict(X_test_raw)
evaluate_model("SVM_RawPixels", y_test_cls, pred_svm_raw)

In [ ]:
# =========================
# 13) Experiment 2: SVM_PCA
# Problem 1: ลดมิติข้อมูลด้วย PCA
# =========================
svm_pca = SVC(kernel="rbf", C=5, gamma="scale", random_state=SEED)
svm_pca.fit(X_train_pca, y_train_cls)

pred_svm_pca = svm_pca.predict(X_test_pca)
evaluate_model("SVM_PCA", y_test_cls, pred_svm_pca)


In [ ]:
# =========================
# 14) Experiment 3: SVM_HOG
# Problem 2: ใช้รูปร่างวัตถุผ่าน HOG
# =========================
svm_hog = SVC(kernel="rbf", C=5, gamma="scale", random_state=SEED)
svm_hog.fit(X_train_hog, y_train_cls)

pred_svm_hog = svm_hog.predict(X_test_hog)
evaluate_model("SVM_HOG", y_test_cls, pred_svm_hog)

In [ ]:
# =========================
# 15) Experiment 4: RF_RawPixels
# Problem 4: baseline อีกแบบด้วย Random Forest
# =========================
rf_raw = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1
)
rf_raw.fit(X_train_raw, y_train_cls)

pred_rf_raw = rf_raw.predict(X_test_raw)
evaluate_model("RF_RawPixels", y_test_cls, pred_rf_raw)


In [ ]:
# =========================
# 16) Experiment 5: RF_HOG
# Problem 2: ใช้ feature ด้านรูปร่าง
# =========================
rf_hog = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1
)
rf_hog.fit(X_train_hog, y_train_cls)

pred_rf_hog = rf_hog.predict(X_test_hog)
evaluate_model("RF_HOG", y_test_cls, pred_rf_hog)

In [ ]:
# =========================
# 17) Experiment 6: RF_ColorHist
# Problem 3: ใช้สีเพื่อช่วยแยกบางคลาส
# =========================
rf_color = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1
)
rf_color.fit(X_train_color, y_train_cls)

pred_rf_color = rf_color.predict(X_test_color)
evaluate_model("RF_ColorHist", y_test_cls, pred_rf_color)

In [ ]:
# =========================
# 18) ฟังก์ชันสร้าง CNN
# ใช้โครงสร้างเดียวกันเพื่อให้เปรียบเทียบยุติธรรม
# =========================
def build_cnn_model(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
# =========================
# 19) เตรียมข้อมูลสำหรับ CNN (ใช้ subset เดียวกับ SVM / RF)
# =========================
y_train_cat = to_categorical(y_train_cls, 10)
y_val_cat   = to_categorical(y_val_cls, 10)
y_test_cat  = to_categorical(y_test_cls, 10)

x_train_cnn_base = x_train_cls.astype("float32")
x_val_cnn_base   = x_val_cls.astype("float32")
x_test_cnn_base  = x_test_cls.astype("float32")

x_train_cnn_norm = x_train_cls.astype("float32") / 255.0
x_val_cnn_norm   = x_val_cls.astype("float32") / 255.0
x_test_cnn_norm  = x_test_cls.astype("float32") / 255.0

print("CNN Train:", x_train_cnn_base.shape, y_train_cat.shape)
print("CNN Val  :", x_val_cnn_base.shape, y_val_cat.shape)
print("CNN Test :", x_test_cnn_base.shape, y_test_cat.shape)

In [ ]:
# =========================
# 20) ฟังก์ชันแสดง learning curve
# =========================
def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(hist["loss"], label="train_loss")
    plt.plot(hist["val_loss"], label="val_loss")
    plt.title(f"{title} - Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist["accuracy"], label="train_acc")
    plt.plot(hist["val_accuracy"], label="val_acc")
    plt.title(f"{title} - Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# 21) Experiment 7: CNN_Baseline
# Problem 5: ใช้ CNN เพื่อเรียนรู้ feature อัตโนมัติ
# =========================
cnn_baseline = build_cnn_model()

history_baseline = cnn_baseline.fit(
    x_train_cnn_base, y_train_cat,
    validation_data=(x_val_cnn_base, y_val_cat),
    epochs=25,
    batch_size=64,
    verbose=1
)

plot_history(history_baseline, "CNN_Baseline")

pred_cnn_baseline = np.argmax(cnn_baseline.predict(x_test_cnn_base), axis=1)
evaluate_model("CNN_Baseline", y_test_cls, pred_cnn_baseline)

In [ ]:
# =========================
# 22) Experiment 8: CNN_Normalize
# Problem 6: ปรับ scale ของ pixel ให้เหมาะกับ neural network
# =========================
cnn_normalize = build_cnn_model()

history_normalize = cnn_normalize.fit(
    x_train_cnn_norm, y_train_cat,
    validation_data=(x_val_cnn_norm, y_val_cat),
    epochs=25,
    batch_size=64,
    verbose=1
)

plot_history(history_normalize, "CNN_Normalize")

pred_cnn_normalize = np.argmax(cnn_normalize.predict(x_test_cnn_norm), axis=1)
evaluate_model("CNN_Normalize", y_test_cls, pred_cnn_normalize)

In [ ]:
# =========================
# 23) Experiment 9: CNN_FlipShift
# Problem 7: ตำแหน่งวัตถุไม่คงที่
# ใช้ Data Augmentation เช่น flip และ random shift/crop
# =========================
datagen = ImageDataGenerator(
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(x_train_cnn_norm)

cnn_flipcrop = build_cnn_model()

history_flipcrop = cnn_flipcrop.fit(
    datagen.flow(x_train_cnn_norm, y_train_cat, batch_size=64),
    validation_data=(x_val_cnn_norm, y_val_cat),
    epochs=25,
    verbose=1
)

plot_history(history_flipcrop, "CNN_FlipCrop")

pred_cnn_flipcrop = np.argmax(cnn_flipcrop.predict(x_test_cnn_norm), axis=1)
evaluate_model("CNN_FlipCrop", y_test_cls, pred_cnn_flipcrop)

In [ ]:
# =========================
# 24) สรุปผลการทดลองทั้งหมด
# =========================
import pandas as pd

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(results_df)

plt.figure(figsize=(12, 5))
plt.bar(results_df["Experiment"], results_df["Accuracy"])
plt.xticks(rotation=45)
plt.ylabel("Accuracy")
plt.title("Comparison of All Experiments")
plt.show()


# ROUND 2: ปรับปรุงทุกโมเดล เพื่อเปรียบเทียบกับรอบแรก
#
 แนวคิด:
- ไม่แก้โค้ดเดิม
- เพิ่ม experiment ใหม่ต่อท้าย
- ตั้งชื่อใหม่ทุกตัวลงท้ายด้วย _Tuned_v2
- คอมเมนต์ชัดเจนว่าปรับอะไร


In [ ]:
# =========================
# 27) ROUND 2 - เตรียม Standardization สำหรับ SVM
#
# สิ่งที่ปรับ:
# 1. เพิ่มการ scale ข้อมูลด้วย StandardScaler
#    เพราะ SVM มักทำงานได้ดีขึ้นเมื่อ feature อยู่ใน scale ที่เหมาะสม
# 2. แยก scaler ตามชนิด feature เพื่อให้เปรียบเทียบได้ชัดเจน
# =========================
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Raw Pixels
scaler_raw = StandardScaler()
X_train_raw_scaled = scaler_raw.fit_transform(X_train_raw)
X_val_raw_scaled   = scaler_raw.transform(X_val_raw)
X_test_raw_scaled  = scaler_raw.transform(X_test_raw)

# PCA
scaler_pca = StandardScaler()
X_train_pca_scaled = scaler_pca.fit_transform(X_train_pca)
X_val_pca_scaled   = scaler_pca.transform(X_val_pca)
X_test_pca_scaled  = scaler_pca.transform(X_test_pca)

# HOG
scaler_hog = StandardScaler()
X_train_hog_scaled = scaler_hog.fit_transform(X_train_hog)
X_val_hog_scaled   = scaler_hog.transform(X_val_hog)
X_test_hog_scaled  = scaler_hog.transform(X_test_hog)

In [ ]:
# =========================
# 28) Experiment 10: SVM_RawPixels_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่ม StandardScaler
# - ปรับ C จาก 5 -> 10 เพื่อให้ model fit decision boundary ได้ยืดหยุ่นขึ้น
# =========================
svm_raw_tuned_v2 = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    random_state=SEED
)

svm_raw_tuned_v2.fit(X_train_raw_scaled, y_train_cls)
pred_svm_raw_tuned_v2 = svm_raw_tuned_v2.predict(X_test_raw_scaled)
evaluate_model("SVM_RawPixels_Tuned_v2", y_test_cls, pred_svm_raw_tuned_v2)

In [ ]:
# =========================
# 29) Experiment 11: SVM_PCA_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่ม StandardScaler หลัง PCA
# - ปรับ C จาก 5 -> 10
# - PCA ช่วยลดมิติ และ scaling ช่วยให้ SVM แยกคลาสได้ดีขึ้น
# =========================
svm_pca_tuned_v2 = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    random_state=SEED
)

svm_pca_tuned_v2.fit(X_train_pca_scaled, y_train_cls)
pred_svm_pca_tuned_v2 = svm_pca_tuned_v2.predict(X_test_pca_scaled)
evaluate_model("SVM_PCA_Tuned_v2", y_test_cls, pred_svm_pca_tuned_v2)

In [ ]:
# =========================
# 30) Experiment 12: SVM_HOG_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่ม StandardScaler
# - ปรับ C จาก 5 -> 10
# - HOG เป็น feature ที่เหมาะกับ SVM อยู่แล้ว
#   เมื่อ scale เพิ่มเติม มักช่วยให้ผลดีขึ้นอีก
# =========================
svm_hog_tuned_v2 = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    random_state=SEED
)

svm_hog_tuned_v2.fit(X_train_hog_scaled, y_train_cls)
pred_svm_hog_tuned_v2 = svm_hog_tuned_v2.predict(X_test_hog_scaled)
evaluate_model("SVM_HOG_Tuned_v2", y_test_cls, pred_svm_hog_tuned_v2)

In [ ]:
# =========================
# 31) Experiment 13: RF_RawPixels_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่มจำนวนต้นไม้จาก 200 -> 500
# - กำหนด max_features='sqrt' เพื่อลดความคล้ายกันของแต่ละต้น
# - กำหนด min_samples_leaf=2 เพื่อลด overfitting
# =========================
rf_raw_tuned_v2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)

rf_raw_tuned_v2.fit(X_train_raw, y_train_cls)
pred_rf_raw_tuned_v2 = rf_raw_tuned_v2.predict(X_test_raw)
evaluate_model("RF_RawPixels_Tuned_v2", y_test_cls, pred_rf_raw_tuned_v2)

In [ ]:
# =========================
# 32) Experiment 14: RF_HOG_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่มจำนวนต้นไม้จาก 300 -> 500
# - ใช้ max_features='sqrt'
# - ใช้ min_samples_leaf=2
# - HOG ช่วยให้ RF เห็น pattern ด้าน shape ชัดกว่าพิกเซลดิบ
# =========================
rf_hog_tuned_v2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)

rf_hog_tuned_v2.fit(X_train_hog, y_train_cls)
pred_rf_hog_tuned_v2 = rf_hog_tuned_v2.predict(X_test_hog)
evaluate_model("RF_HOG_Tuned_v2", y_test_cls, pred_rf_hog_tuned_v2)

In [ ]:
# =========================
# 33) Experiment 15: RF_ColorHist_Tuned_v2
#
# สิ่งที่ปรับจากเดิม:
# - เพิ่มจำนวนต้นไม้
# - ใช้ max_features='sqrt'
# - ใช้ min_samples_leaf=2
# หมายเหตุ:
# - Color Histogram มีข้อมูลเฉพาะเรื่องสี จึงอาจยังสู้ HOG หรือ CNN ไม่ได้
# =========================
rf_color_tuned_v2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)

rf_color_tuned_v2.fit(X_train_color, y_train_cls)
pred_rf_color_tuned_v2 = rf_color_tuned_v2.predict(X_test_color)
evaluate_model("RF_ColorHist_Tuned_v2", y_test_cls, pred_rf_color_tuned_v2)

In [ ]:
# =========================
# ROUND 2 - CNN ทั้ง 3 แบบให้แฟร์
# ใช้ architecture / epochs / batch_size / callbacks เดียวกัน
# ต่างกันเฉพาะ input preprocessing และ augmentation
# =========================

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# callbacks กลาง ใช้เหมือนกันทั้ง 3 โมเดล
callbacks_v2 = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )
]

EPOCHS_V2 = 50
BATCH_SIZE_V2 = 64

In [ ]:
# -------------------------
# CNN 1: Baseline_Tuned_v2
# ไม่ normalize
# ไม่ augmentation
# -------------------------
cnn_baseline_tuned_v2 = build_cnn_model()

history_baseline_tuned_v2 = cnn_baseline_tuned_v2.fit(
    x_train_cnn_base,
    y_train_cat,
    validation_data=(x_val_cnn_base, y_val_cat),
    epochs=EPOCHS_V2,
    batch_size=BATCH_SIZE_V2,
    verbose=1,
    callbacks=callbacks_v2
)

plot_history(history_baseline_tuned_v2, "CNN_Baseline_Tuned_v2")

pred_cnn_baseline_tuned_v2 = np.argmax(
    cnn_baseline_tuned_v2.predict(x_test_cnn_base),
    axis=1
)

evaluate_model(
    "CNN_Baseline_Tuned_v2",
    y_test_cls,
    pred_cnn_baseline_tuned_v2
)


In [ ]:
# -------------------------
# CNN 2: Normalize_Tuned_v2
# normalize อย่างเดียว
# ไม่ augmentation
# -------------------------
cnn_normalize_tuned_v2 = build_cnn_model()

history_normalize_tuned_v2 = cnn_normalize_tuned_v2.fit(
    x_train_cnn_norm,
    y_train_cat,
    validation_data=(x_val_cnn_norm, y_val_cat),
    epochs=EPOCHS_V2,
    batch_size=BATCH_SIZE_V2,
    verbose=1,
    callbacks=callbacks_v2
)

plot_history(history_normalize_tuned_v2, "CNN_Normalize_Tuned_v2")

pred_cnn_normalize_tuned_v2 = np.argmax(
    cnn_normalize_tuned_v2.predict(x_test_cnn_norm),
    axis=1
)

evaluate_model(
    "CNN_Normalize_Tuned_v2",
    y_test_cls,
    pred_cnn_normalize_tuned_v2
)

In [ ]:
# -------------------------
# CNN 3: AdvancedAugmentation_Tuned_v2
# normalize + augmentation แบบปรับปรุงจากรอบ 1
# จุดต่างจากรอบ 1:
# 1) ยังใช้ flip + shift เหมือนเดิม
# 2) เพิ่ม zoom แบบเบา เพื่อให้ model ทนต่อขนาดวัตถุที่ต่างกัน
# 3) ใช้ callbacks และ epochs มากขึ้นแบบเดียวกับ CNN_Tuned_v2 อื่น ๆ
# 4) กำหนด steps_per_epoch ให้ชัดเจน
# 5) ลดความแรง augmentation เพื่อให้เหมาะกับภาพขนาด 32x32
# -------------------------
import math

datagen_tuned_v2 = ImageDataGenerator(
    horizontal_flip=True,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.03
)

train_generator_v2 = datagen_tuned_v2.flow(
    x_train_cnn_norm,
    y_train_cat,
    batch_size=BATCH_SIZE_V2,
    shuffle=True,
    seed=SEED
)

cnn_tuned_v2 = build_cnn_model()

history_tuned_v2 = cnn_tuned_v2.fit(
    train_generator_v2,
    validation_data=(x_val_cnn_norm, y_val_cat),
    epochs=EPOCHS_V2,
    steps_per_epoch=math.ceil(len(x_train_cnn_norm) / BATCH_SIZE_V2),
    verbose=1,
    callbacks=callbacks_v2
)

plot_history(history_tuned_v2, "CNN_AdvancedAugmentation_Tuned_v2")

pred_cnn_tuned_v2 = np.argmax(
    cnn_tuned_v2.predict(x_test_cnn_norm, verbose=0),
    axis=1
)

evaluate_model(
    "CNN_AdvancedAugmentation_Tuned_v2",
    y_test_cls,
    pred_cnn_tuned_v2
)

In [ ]:
# =========================
# สรุปผล Round 2 เท่านั้น
# =========================
import pandas as pd

if "results" not in globals():
    results = []

results_df = pd.DataFrame(results).copy()

if len(results_df) == 0:
    print("ยังไม่มีผลลัพธ์ในตัวแปร results")
else:
    results_df = results_df.drop_duplicates(subset=["Experiment"], keep="last").reset_index(drop=True)

    round2_df = results_df[results_df["Experiment"].str.contains("Tuned_v2", na=False)].copy()

    if len(round2_df) == 0:
        print("ยังไม่มีผลลัพธ์ของ Round 2")
    else:
        round2_df = round2_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

        print("\n===== Summary: Round 2 =====")
        print(round2_df[["Experiment", "Accuracy", "Precision", "Recall", "F1-Score"]])

        best_r2 = round2_df.iloc[0]
        print("\n===== Best Model in Round 2 =====")
        print(f"Experiment: {best_r2['Experiment']}")
        print(f"Accuracy  : {best_r2['Accuracy']:.4f}")
        print(f"Precision : {best_r2['Precision']:.4f}")
        print(f"Recall    : {best_r2['Recall']:.4f}")
        print(f"F1-Score  : {best_r2['F1-Score']:.4f}")